In [3]:
import json

# Paths to your two input files
# FILE1 should contain metadata for each record:
#   { "ID1": { "title": ..., "author": ..., ... }, ... }
# FILE2 should contain entities for each record:
#   { "ID1": { "entities": [ ... ] }, ... }
FILE1 = r"C:\Users\samue\OneDrive\Desktop\Test_Data-20250428T160935Z-001\Test_Data\articles_test.json"
FILE2 = r"C:\Users\samue\OneDrive\Desktop\ataupd2425-gainer_T61_ms_trainplatinumgoldsilver\ataupd2425-gainer_T61_ms_trainplatinumgoldsilver.json"
OUTPUT = r"C:\Users\samue\OneDrive\Desktop\mergedms_trainplatinumgoldsilver.json"

def load_json(path):
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f)

def merge_files(meta, ent):
    merged = {}
    for record_id, md in meta.items():
        merged[record_id] = {
            'metadata': md,
            # if there are entities for this id, use them; else empty list
            'entities': ent.get(record_id, {}).get('entities', [])
        }
    return merged

def main():
    meta_data = load_json(FILE1)
    ent_data = load_json(FILE2)

    merged = merge_files(meta_data, ent_data)

    with open(OUTPUT, 'w', encoding='utf-8') as f:
        json.dump(merged, f, ensure_ascii=False, indent=4)

    print(f"Merged {len(merged)} records into {OUTPUT}")

if __name__ == '__main__':
    main()


Merged 40 records into C:\Users\samue\OneDrive\Desktop\mergedms_trainplatinumgoldsilver.json


In [3]:
import os
import re

def validate_run_folders(base_path):
    """
    Validate GutBrainIE CLEF2025 submission folder structure under `base_path`.

    Returns True if all checks pass, False otherwise.
    """
    # 1. Check base_path
    if not os.path.isdir(base_path):
        print(f"Error: '{base_path}' is not a valid directory.")
        return False

    # 2. Gather subfolders
    subdirs = [
        d for d in os.listdir(base_path)
        if os.path.isdir(os.path.join(base_path, d))
    ]

    # 3. Pattern: TeamID_TaskID_RunID[_SystemDesc]
    pattern = re.compile(r'^([^_]+)_(T6(?:21|22|23|1))_([^_]+)(?:_(.+))?$')

    parsed = []
    errors = []

    for d in subdirs:
        m = pattern.match(d)
        if not m:
            errors.append(
                f"Folder '{d}' does not follow "
                "'<TeamID>_<TaskID>_<RunID>[_<SystemDesc>]' pattern."
            )
        else:
            team, task, run_id, system = m.groups()
            parsed.append({
                "folder": d,
                "team": team,
                "task": task,
                "run": run_id,
                "system": system or None
            })

    # 4. Early exit on naming errors
    if errors:
        for e in errors:
            print("Error:", e)
        return False

    if not parsed:
        print("Error: No subfolders matching the naming convention were found.")
        return False

    # 5. Single TeamID check
    teams = {p["team"] for p in parsed}
    if len(teams) > 1:
        print("Error: Multiple TeamIDs found:", ", ".join(sorted(teams)))
        return False
    team = teams.pop()

    # 6. Count tasks
    task_counts = {}
    for p in parsed:
        task_counts[p["task"]] = task_counts.get(p["task"], 0) + 1

    # 7. Collect runs & systems
    runs = sorted({p["run"] for p in parsed})
    systems = sorted({s for s in (p["system"] for p in parsed) if s})

    # 8. Print summary
    print(f"Team ID: {team}")
    print("Tasks:")
    for t in sorted(task_counts, key=lambda x: int(x.replace('T6', ''))):
        cnt = task_counts[t]
        print(f"  {t}: {cnt} folder{'s' if cnt != 1 else ''}")
    print("Runs:", ", ".join(runs))
    print("System Descriptions:", ", ".join(systems) if systems else "None")

    # 9. Verify per‐folder JSON + META presence
    all_ok = True
    for p in parsed:
        fp = os.path.join(base_path, p["folder"])
        need = [f"{p['folder']}.json", f"{p['folder']}.meta"]
        missing = [n for n in need if n not in os.listdir(fp)]
        if missing:
            print(f"Error in '{p['folder']}': missing {', '.join(missing)}")
            all_ok = False

    if all_ok:
        print("All folders and files are valid.")
    return all_ok

# Example usage in a notebook cell:
base_path = r"C:\Users\samue\OneDrive\Desktop\ataupd2425-gainer_T61_ma_trainplatinumandgold"
validate_run_folders(base_path)


Error: No subfolders matching the naming convention were found.


False